In [ ]:
import json
import argparse
import numpy as np
import torch
import matplotlib.pyplot as plt
import shutil
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scripts.video_results_scripts import (plot_general_training_info, plot_ROC_curves, find_best_threshold_bacc,
                                          confusion_matrix_plot)
from sklearn.metrics import accuracy_score, precision_score, balanced_accuracy_score
from sklearn.metrics import balanced_accuracy_score, average_precision_score

In [38]:
results_file = 'evid_bbloss_wkl_wprior_6enc_temporal_2_results.json'
PWD = Path.cwd()
results_dir = PWD / 'results'
with open(results_dir / results_file, 'r') as f:
    results = json.load(f)

train_avg_bal_acc = []
val_avg_bal_acc = []
for key in results.keys():
    if 'Train' in key:
        train_avg_bal_acc.append(results[key]['avg_bacc'])
    if 'Val' in key:
        val_avg_bal_acc.append(results[key]['avg_bacc'])
# Train
best_train_index = np.argmax(train_avg_bal_acc)
train_epochs = [x for x in list(results.keys()) if 'Train' in x]
train_results = results[train_epochs[best_train_index]]
# Val
best_val_index = np.argmax(val_avg_bal_acc)
val_epochs = [x for x in list(results.keys()) if 'Val' in x]
val_results = results[val_epochs[best_val_index]]
# Test
test_results = results[list(results.keys())[-1]]

data = {
"Split\Metric": ["Epoch", "Accuracy", "Balanced Acc.", "mAP"],
"Train": [int(best_train_index+1), 100*train_results['avg_accuracy'], 100*train_results['avg_bacc'], 100*train_results['mAP']],
"Val":   [int(best_val_index+1),   100*val_results['avg_accuracy'],   100*val_results['avg_bacc'],   100*val_results['mAP']],
"Test":  [int(best_val_index+1),   100*test_results['avg_accuracy'],  100*test_results['avg_bacc'],  100*test_results['mAP']]
}
basic_results = pd.DataFrame(data).set_index("Split\Metric").T
print(f"Basic results")
print(basic_results)

Basic results
Split\Metric  Epoch  Accuracy  Balanced Acc.    mAP
Train           8.0     96.88          96.85  93.01
Val             6.0     87.66          73.59  61.00
Test            6.0     83.42          73.02  66.09


In [41]:
"""
FAST Video-level evaluation (VAL + TEST) with two runs.

RUN 1 (fixed baseline):
  - Uses a fixed frame threshold of 0.5 for all classes (p only)
  - Aggregates to video level via OR
  - Prints counts + metrics (VAL + TEST)

RUN 2 (your ask: "precision=1 across videos", no uncertainty):
  - Find the MOST LIBERAL threshold T on VAL such that *video-level* precision == 1.0
    (i.e., precision >= 1.0 with float tolerance)
  - Report that T (per class)
  - Apply that same T to TEST and print counts + metrics

Notes:
  - "Most liberal" == lowest threshold that still achieves video-precision 1.
  - If no threshold can achieve precision 1 on VAL (with non-zero predictions),
    we report that explicitly and fall back to the best-achievable precision (then lowest T).
"""

import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, balanced_accuracy_score


# =============================================================================
# Utility: make contiguous 0..(n_vids-1) video indices (scatter_reduce needs this)
# =============================================================================
def remap_vid_ids_to_contiguous(vid_ids: torch.Tensor):
    vid_ids = vid_ids.detach().flatten().cpu().to(torch.long)
    uniq_vids, inv = torch.unique(vid_ids, sorted=True, return_inverse=True)
    return inv.to(torch.long), uniq_vids


# =============================================================================
# FAST OR-aggregation using scatter_reduce_
# =============================================================================
def video_or_from_frame_pred(
    frame_gt: torch.Tensor,    # int64 [N] in {0,1}
    frame_pred: torch.Tensor,  # int64 [N] in {0,1}
    vid_idx: torch.Tensor,     # int64 [N] in 0..n_vids-1
    n_vids: int,
):
    frame_gt = frame_gt.detach().flatten().cpu().to(torch.int64)
    frame_pred = frame_pred.detach().flatten().cpu().to(torch.int64)
    vid_idx = vid_idx.detach().flatten().cpu().to(torch.long)

    y_video = torch.zeros(n_vids, dtype=torch.int64)
    p_video = torch.zeros(n_vids, dtype=torch.int64)

    # OR == max for {0,1}
    y_video.scatter_reduce_(0, vid_idx, frame_gt, reduce="amax", include_self=True)
    p_video.scatter_reduce_(0, vid_idx, frame_pred, reduce="amax", include_self=True)

    return y_video.numpy(), p_video.numpy()


def video_metrics_from_frame_pred_fast(
    y_frame: torch.Tensor,     # float [N], soft labels
    frame_pred: torch.Tensor,  # int64 [N] in {0,1}
    vid_ids: torch.Tensor,     # [N]
):
    y_frame = y_frame.detach().flatten().cpu()
    frame_pred = frame_pred.detach().flatten().cpu().to(torch.int64)

    vid_idx, uniq_vids = remap_vid_ids_to_contiguous(vid_ids)
    n_vids = int(uniq_vids.numel())

    frame_gt = (y_frame >= 0.5).to(torch.int64)
    y_video, p_video = video_or_from_frame_pred(frame_gt, frame_pred, vid_idx, n_vids)

    acc = accuracy_score(y_video, p_video)
    prec = precision_score(y_video, p_video, zero_division=0)
    rec = recall_score(y_video, p_video, zero_division=0)
    bacc = balanced_accuracy_score(y_video, p_video)

    return {
        "n_vids": n_vids,
        "gt_pos": int(y_video.sum()),
        "pred_pos": int(p_video.sum()),
        "acc": float(acc),
        "prec": float(prec),
        "rec": float(rec),
        "bacc": float(bacc),
    }


def video_metrics_from_frame_scores_fast(
    y_frame: torch.Tensor,
    score_frame: torch.Tensor,
    vid_ids: torch.Tensor,
    thresh: float,
):
    frame_pred = (score_frame.detach().flatten().cpu() >= float(thresh)).to(torch.int64)
    return video_metrics_from_frame_pred_fast(y_frame, frame_pred, vid_ids)


def eval_and_print_video_metrics_fast(
    split_name: str,
    label: str,
    y_frame: torch.Tensor,
    score_frame: torch.Tensor,
    vid_ids: torch.Tensor,
    thresh: float,
    note: str = "",
):
    m = video_metrics_from_frame_scores_fast(y_frame, score_frame, vid_ids, thresh)
    print(f"\n=== {label} | {split_name} | video-level ==={(' ' + note) if note else ''}")
    print(f"thresh={float(thresh):.6f}")
    print(f"Videos: {m['n_vids']} | GT positives: {m['gt_pos']} | Pred positives: {m['pred_pos']}")
    #print(f"Accuracy:          {m['acc']:.4f}")
    print(f"Precision:         {m['prec']:.4f}")
    print(f"Recall:            {m['rec']:.4f}")
    #print(f"Balanced accuracy: {m['bacc']:.4f}")
    return m


def print_split_stats_fast(split_name: str, results_dict, class_keys=("C1", "C2", "C3")):
    vid_ids = torch.tensor(results_dict["saved"]["vid_ids"]).detach().flatten().cpu().to(torch.long)
    vid_idx, uniq_vids = remap_vid_ids_to_contiguous(vid_ids)
    n_vids = int(uniq_vids.numel())

    labels = torch.tensor(results_dict["saved"]["labels"], dtype=torch.float32).detach().cpu()

    print(f"\n[{split_name}] Videos: {n_vids}")
    for i, key in enumerate(class_keys):
        y_frame = labels[:, i]
        frame_gt = (y_frame >= 0.5).to(torch.int64)
        y_video = torch.zeros(n_vids, dtype=torch.int64)
        y_video.scatter_reduce_(0, vid_idx, frame_gt, reduce="amax", include_self=True)
        print(f"  {key}: GT positives (video-level) = {int(y_video.sum().item())}")


# =============================================================================
# RUN 2 helper: find MOST LIBERAL threshold achieving video precision == 1.0
# =============================================================================
def find_most_liberal_T_precision1_video_or_fast(
    y_frame: torch.Tensor,
    p_frame: torch.Tensor,
    vid_ids: torch.Tensor,
    thresholds: torch.Tensor | None = None,
    precision_target: float = 1.0,
    eps: float = 1e-12,
):
    """
    Returns dict with chosen threshold + metrics:
      - Chooses the LOWEST threshold T such that video precision >= precision_target.
      - Ignores candidates where pred_pos == 0 (precision would be zero_division=0 => 0).
    If impossible, returns None.
    """
    y_frame = y_frame.detach().flatten().cpu()
    p_frame = p_frame.detach().flatten().cpu()
    vid_ids = vid_ids.detach().flatten().cpu()

    if thresholds is None:
        # Unique scores can be large; pass a grid for speed if desired.
        thresholds = torch.unique(p_frame).sort().values
        thresholds = torch.cat([torch.tensor([0.0]), thresholds, torch.tensor([1.0])]).unique().sort().values
    else:
        thresholds = thresholds.detach().flatten().cpu().sort().values

    best = None
    for t in thresholds:
        t_f = float(t.item())
        m = video_metrics_from_frame_scores_fast(y_frame, p_frame, vid_ids, t_f)

        # Need precision==1 AND at least one predicted positive (else trivial empty)
        if m["pred_pos"] == 0:
            continue
        if m["prec"] + eps < precision_target:
            continue

        # MOST LIBERAL == minimal threshold
        if best is None or t_f < best["T"]:
            best = {"T": t_f, **m}

    return best


def find_best_achievable_precision_video_or_fast(
    y_frame: torch.Tensor,
    p_frame: torch.Tensor,
    vid_ids: torch.Tensor,
    thresholds: torch.Tensor,
):
    """
    Fallback when precision=1 cannot be achieved:
      - maximise precision
      - tie-breaker: maximise recall
      - tie-breaker: choose lower threshold (more liberal)
    """
    y_frame = y_frame.detach().flatten().cpu()
    p_frame = p_frame.detach().flatten().cpu()
    vid_ids = vid_ids.detach().flatten().cpu()
    thresholds = thresholds.detach().flatten().cpu().sort().values

    best = None
    for t in thresholds:
        t_f = float(t.item())
        m = video_metrics_from_frame_scores_fast(y_frame, p_frame, vid_ids, t_f)

        # You can decide whether to allow pred_pos==0; here we allow it but it will rarely win.
        if best is None:
            best = {"T": t_f, **m}
            continue

        better = (
            (m["prec"] > best["prec"])
            or (m["prec"] == best["prec"] and m["rec"] > best["rec"])
            or (m["prec"] == best["prec"] and m["rec"] == best["rec"] and t_f < best["T"])
        )
        if better:
            best = {"T": t_f, **m}

    return best


# =============================================================================
# Main evaluation driver (adapted to your requirements)
# =============================================================================
def run_video_eval_fast_precision1_only(
    val_results,
    test_results,
    class_keys=("C1", "C2", "C3"),
    # Run 1 fixed baseline threshold
    run1_thresh: float = 0.5,
    # Run 2 search controls
    use_grid_for_T: bool = True,
    T_grid_size: int = 1501,
):
    # Split stats
    print_split_stats_fast("VAL", val_results, class_keys)
    print_split_stats_fast("TEST", test_results, class_keys)

    # -------------------------
    # RUN 1 — Fixed baseline (0.5)
    # -------------------------
    print("\n" + "=" * 80)
    print(f"RUN 1 — Baseline: fixed threshold p >= {run1_thresh:.2f} (p only)")
    print("=" * 80)

    for label_idx, label in enumerate(class_keys):
        # VAL
        p_val = torch.tensor(val_results["saved"][label]["probs"], dtype=torch.float32)
        y_val = torch.tensor(val_results["saved"]["labels"], dtype=torch.float32)[:, label_idx]
        vid_val = torch.tensor(val_results["saved"]["vid_ids"], dtype=torch.int64)
        eval_and_print_video_metrics_fast("VAL", label, y_val, p_val, vid_val, thresh=run1_thresh)

        # TEST
        p_test = torch.tensor(test_results["saved"][label]["probs"], dtype=torch.float32)
        y_test = torch.tensor(test_results["saved"]["labels"], dtype=torch.float32)[:, label_idx]
        vid_test = torch.tensor(test_results["saved"]["vid_ids"], dtype=torch.int64)
        eval_and_print_video_metrics_fast("TEST", label, y_test, p_test, vid_test, thresh=run1_thresh)

    # -------------------------
    # RUN 2 — Find MOST LIBERAL threshold with video precision 1 on VAL
    # -------------------------
    print("\n" + "=" * 80)
    print("RUN 2 — p-only: find MOST LIBERAL threshold T on VAL with *video precision == 1.0*")
    print("(No uncertainty, no two-tier rule.)")
    print("=" * 80)

    for label_idx, label in enumerate(class_keys):
        p_val = torch.tensor(val_results["saved"][label]["probs"], dtype=torch.float32)
        y_val = torch.tensor(val_results["saved"]["labels"], dtype=torch.float32)[:, label_idx]
        vid_val = torch.tensor(val_results["saved"]["vid_ids"], dtype=torch.int64)

        p_test = torch.tensor(test_results["saved"][label]["probs"], dtype=torch.float32)
        y_test = torch.tensor(test_results["saved"]["labels"], dtype=torch.float32)[:, label_idx]
        vid_test = torch.tensor(test_results["saved"]["vid_ids"], dtype=torch.int64)

        print(f"\n--- {label} ---")

        thresholds = None
        if use_grid_for_T:
            thresholds = torch.linspace(0.0, 1.0, T_grid_size)

        best = find_most_liberal_T_precision1_video_or_fast(
            y_frame=y_val,
            p_frame=p_val,
            vid_ids=vid_val,
            thresholds=thresholds,
            precision_target=1.0,
        )

        if best is None:
            print("WARNING: Could not achieve video precision=1.0 on VAL (with any non-empty predictions) "
                  "within the searched thresholds.")
            th = thresholds if thresholds is not None else torch.unique(p_val).sort().values
            fb = find_best_achievable_precision_video_or_fast(y_val, p_val, vid_val, th)

            print(
                f"Fallback (best achievable on VAL): "
                f"T={fb['T']:.6f} | Prec={fb['prec']:.4f} Rec={fb['rec']:.4f} "
                f"Pred+={fb['pred_pos']} GT+={fb['gt_pos']}"
            )

            eval_and_print_video_metrics_fast("VAL", label, y_val, p_val, vid_val, thresh=fb["T"], note="(fallback)")
            eval_and_print_video_metrics_fast("TEST", label, y_test, p_test, vid_test, thresh=fb["T"], note="(fallback)")
            continue

        T = best["T"]
        print(
            f"Chosen (most liberal) on VAL with Prec=1.0: "
            f"T={T:.6f} | Prec={best['prec']:.4f} Rec={best['rec']:.4f} "
            f"Pred+={best['pred_pos']} GT+={best['gt_pos']}"
        )

        # Apply to VAL + TEST for reporting
        eval_and_print_video_metrics_fast("VAL", label, y_val, p_val, vid_val, thresh=T, note="(Prec=1.0 threshold)")
        eval_and_print_video_metrics_fast("TEST", label, y_test, p_test, vid_test, thresh=T, note="(applied from VAL)")


# =============================================================================
# Example call:
# =============================================================================
run_video_eval_fast_precision1_only(
    val_results, test_results,
    class_keys=("C1","C2","C3"),
    run1_thresh=0.5,
    use_grid_for_T=True, T_grid_size=1501
)


[VAL] Videos: 41
  C1: GT positives (video-level) = 22
  C2: GT positives (video-level) = 22
  C3: GT positives (video-level) = 20

[TEST] Videos: 40
  C1: GT positives (video-level) = 25
  C2: GT positives (video-level) = 20
  C3: GT positives (video-level) = 23

RUN 1 — Baseline: fixed threshold p >= 0.50 (p only)

=== C1 | VAL | video-level ===
thresh=0.500000
Videos: 41 | GT positives: 22 | Pred positives: 35
Precision:         0.6000
Recall:            0.9545

=== C1 | TEST | video-level ===
thresh=0.500000
Videos: 40 | GT positives: 25 | Pred positives: 35
Precision:         0.6857
Recall:            0.9600

=== C2 | VAL | video-level ===
thresh=0.500000
Videos: 41 | GT positives: 22 | Pred positives: 27
Precision:         0.5926
Recall:            0.7273

=== C2 | TEST | video-level ===
thresh=0.500000
Videos: 40 | GT positives: 20 | Pred positives: 21
Precision:         0.9048
Recall:            0.9500

=== C3 | VAL | video-level ===
thresh=0.500000
Videos: 41 | GT positives: 

In [50]:
for label_idx, label in enumerate(['C1', 'C2', 'C3']):
    u_val = torch.tensor(val_results['saved'][label]["uncerts"], dtype=torch.float32)
    y_val = torch.tensor(val_results['saved']["labels"],  dtype=torch.float32)[:,label_idx]
    pos_mask = y_val > 0.5
    neg_mask = y_val < 0.5
    pos_u_val = u_val[pos_mask]
    neg_u_val = u_val[neg_mask]

    def stats(x):
        if x.numel() == 0:
            return float('nan'), float('nan')
        return x.mean().item(), x.std(unbiased=True).item()

    pos_mean, pos_sd = stats(pos_u_val)
    neg_mean, neg_sd = stats(neg_u_val)

    print(f"{label}")
    print(f"  Positive samples: mean={pos_mean:.4f}, sd={pos_sd:.4f}, n={pos_u_val.numel()}")
    print(f"  Negative samples: mean={neg_mean:.4f}, sd={neg_sd:.4f}, n={neg_u_val.numel()}")
    

C1
  Positive samples: mean=0.1248, sd=0.0439, n=381
  Negative samples: mean=0.1158, sd=0.0309, n=1950
C2
  Positive samples: mean=0.1818, sd=0.0704, n=291
  Negative samples: mean=0.1056, sd=0.0451, n=2040
C3
  Positive samples: mean=0.1935, sd=0.0497, n=389
  Negative samples: mean=0.1621, sd=0.0481, n=1942


In [67]:
for label_idx, label in enumerate(['C1', 'C2', 'C3']):
    print(label)
    p_val = torch.tensor(val_results['saved'][label]["probs"],   dtype=torch.float32)
    u_val = torch.tensor(val_results['saved'][label]["uncerts"], dtype=torch.float32)  # not used for metrics below
    y_val = torch.tensor(val_results['saved']["labels"],         dtype=torch.float32)[:, label_idx]

    p_test = torch.tensor(test_results['saved'][label]["probs"],   dtype=torch.float32)
    u_test = torch.tensor(test_results['saved'][label]["uncerts"], dtype=torch.float32)  # not used for metrics below
    y_test = torch.tensor(test_results['saved']["labels"],         dtype=torch.float32)[:, label_idx]

    bacc_val = balanced_accuracy_score(y_val, p_val >= 0.5)
    print(f"BACC VAL: {round(bacc_val*100,2)}")
    bacc_test = balanced_accuracy_score(y_test, p_test >= 0.5)
    print(f"BACC VAL: {round(bacc_test*100,2)}")

C1
BACC VAL: 75.47
BACC VAL: 72.84
C2
BACC VAL: 74.23
BACC VAL: 67.52
C3
BACC VAL: 69.87
BACC VAL: 72.17


In [25]:
results_file = 'SwinCVS_frozen_ENDP_sd4_results.json'
PWD = Path.cwd()
results_dir = PWD / 'results'
with open(results_dir / results_file, 'r') as f:
    results = json.load(f)

for epoch_name in results.keys():
    if 'Testing' in epoch_name:
        test_results = results[epoch_name]
        print(epoch_name)

y_test = test_results['true']
pred_test = test_results['preds']
prob_test = test_results['preds_prob']
acc_list = []
bacc_list = []
map_list = []
for label_idx, label in enumerate(['C1', 'C2', 'C3']):
    y = np.array(y_test)[:,label_idx]
    preds = np.array(pred_test)[:,label_idx]
    probs = np.array(prob_test)[:,label_idx]
    accuracy = np.mean(y == preds)
    bacc = balanced_accuracy_score(y, preds)
    ap = average_precision_score(y, probs)
    print(label)
    print(round(accuracy,4), round(bacc,4), round(ap,4))

Testing_Epoch_2
C1
0.8035 0.755 0.6552
C2
0.8731 0.6714 0.6145
C3
0.7823 0.6349 0.6784


In [36]:
results_file = f"evid_bbloss_wkl_wprior_6enc_temporal_0_results.json"
PWD = Path.cwd()
results_dir = PWD / 'results'
with open(results_dir / results_file, 'r') as f:
    results = json.load(f)

train_avg_bal_acc = []
val_avg_bal_acc = []
for key in results.keys():
    if 'Train' in key:
        train_avg_bal_acc.append(results[key]['avg_bacc'])
    if 'Val' in key:
        val_avg_bal_acc.append(results[key]['avg_bacc'])
# Train
best_train_index = np.argmax(train_avg_bal_acc)
train_epochs = [x for x in list(results.keys()) if 'Train' in x]
train_results = results[train_epochs[best_train_index]]
# Val
best_val_index = np.argmax(val_avg_bal_acc)
val_epochs = [x for x in list(results.keys()) if 'Val' in x]
val_results = results[val_epochs[best_val_index]]
# Test
test_results = results[list(results.keys())[-1]]

In [15]:
def get_soft_labels_from_annotations(vid_tensor, frame_tensor):
    """
    ann_dict: dictionary with keys ['train','val','test']
    vid_tensor: (N,) tensor of video ids
    frame_tensor: (N,) tensor of frame ids

    returns:
        ds_tensor: (N, 3) tensor with ds values aligned to input order
    """
    with open(PWD / 'config/temporal_annotations.json', 'r') as f:
        ann_dict = json.load(f)
    vid_list = vid_tensor.tolist()
    frame_list = frame_tensor.tolist()

    out = []

    for vid, frame in zip(vid_list, frame_list):

        # --- determine split ---
        if vid <= 120:
            split = 'train'
        elif vid <= 161:
            split = 'val'
        else:
            split = 'test'

        # --- build filename key ---
        fname = f"{int(vid)}_{int(frame)}.jpg"

        try:
            ds = ann_dict[split][fname]['annotations']['ds']
        except KeyError:
            raise KeyError(f"Missing annotation for {split}/{fname}")

        out.append(ds)

    return torch.tensor(out, dtype=torch.float32)

In [37]:
for label_idx, label in enumerate(['C1', 'C2', 'C3']):
    probs = np.array(test_results['saved'][label]['probs'])
    preds = np.array(test_results['saved'][label]['preds'])
    frame_ids = np.array(test_results['saved']['frame_ids'])
    vid_ids = np.array(test_results['saved']['vid_ids'])
    y = np.array(get_soft_labels_from_annotations(vid_ids, frame_ids))[:,label_idx]
    # Mask for y == 0.3333 OR y == 0.6667 (float-safe)
    mask_y = np.isclose(y, 1/3) | np.isclose(y, 2/3)

    probs = probs[mask_y]
    preds = preds[mask_y]
    y = np.round(y[mask_y])

    acc = accuracy_score(y, preds)
    bacc = balanced_accuracy_score(y, preds)
    ap = average_precision_score(y, probs)
    print(label)
    print(f"ACC:    {round(100*acc,2)}")
    print(f"BACC:   {round(100*bacc,2)}")
    print(f"AP:     {round(100*ap,2)}\n")

C1
ACC:    66.19
BACC:   66.0
AP:     70.25

C2
ACC:    59.63
BACC:   56.23
AP:     63.23

C3
ACC:    67.7
BACC:   65.96
AP:     68.9

